# Procesamiento de datos

**Analisis y reconocimiento**


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path


Se importan las librerias que utilizaremos para poder limpiar los datos. Ahora importaremos los archivos.


In [2]:
datos_test = pd.read_excel('data/Lab3_datos_Prueba.xlsx')
datos = pd.read_excel('data/Lab3_Datos.xlsx')
df_diccionario = pd.read_excel('data/Lab3_diccionario_Datos.xlsx')


Ahora si empezaremos con el analisis y solucion respectivamente de cada uno.


In [3]:
df_test = datos_test.copy()
df = datos.copy()
df['id_fila'] = df.index


Realizamos una copia de los datos para evitar problemas con los archivos originales. Agregamos id_fila para conservar el registro de cada fila. Este identificador no se usara como predictor. Ahora vamos a ver la completitud de los datos.


In [4]:
((df.isnull().sum()/df.shape[0])).sort_values(ascending=False)


vehiculo                           0.991472
frecuencia_cafeteria               0.017135
frecuencia_restaurante_20_a_50     0.014924
frecuencia_comida_para_llevar      0.011924
frecuencia_restaurante_menos_20    0.010265
frecuencia_bar                     0.008449
destino                            0.000000
acepta_cupon                       0.000000
direccion_opuesta                  0.000000
misma_direccion                    0.000000
tiempo_cupon_mayor_25min           0.000000
tiempo_cupon_mayor_15min           0.000000
tiempo_cupon_mayor_5min            0.000000
ingreso                            0.000000
acompanante                        0.000000
ocupacion                          0.000000
educacion                          0.000000
tiene_hijos                        0.000000
estado_civil                       0.000000
edad                               0.000000
genero                             0.000000
vigencia_cupon                     0.000000
cupon                           

En este caso tenemos 6 variables que no cuentan con la informacion completa. Ahora vamos a mirar los duplicados.


In [5]:
columnas_comparar = [col for col in df.columns if col != 'id_fila']
n_registros_repetidos = int(df.duplicated(subset=columnas_comparar, keep=False).sum())
n_duplicados = int(df.duplicated(subset=columnas_comparar).sum())
print(f'Registros involucrados en duplicados: {n_registros_repetidos}')
print(f'Copias adicionales: {n_duplicados}')


Registros involucrados en duplicados: 148
Copias adicionales: 74


Hay 148 registros involucrados en duplicados, que corresponden a 74 copias adicionales. Los trataremos mas adelante sin incluir id_fila en la comparacion. Ahora vamos a mirar la consistencia de los datos.


In [6]:
columnas_texto = df.select_dtypes(include=['object', 'string']).columns
for col in columnas_texto:
    print(df[col].value_counts())


Sin lugar urgente    6270
Casa                 3233
Trabajo              3161
Name: destino, dtype: int64
Solo        7296
Amigo(s)    3291
Pareja      1072
Hijo(s)     1005
Name: acompanante, dtype: int64
Soleado     10052
Nevado       1404
Lluvioso     1208
Name: clima, dtype: int64
6:00 p.m.     3224
7:00 a.m.     3161
10:00 a.m.    2271
2:00 p.m.     2005
10:00 p.m.    2003
Name: hora, dtype: int64
Cafetería             3986
Restaurante(<20)      2776
Comida para llevar    2393
Bar                   2017
Restaurante(20-50)    1492
Name: cupon, dtype: int64
1 día      7081
2 horas    5583
Name: vigencia_cupon, dtype: int64
Femenino     6501
Masculino    6163
Name: genero, dtype: int64
21             2652
26             2555
31             2033
50 o más       1785
36             1316
41             1091
46              685
menor de 21     547
Name: edad, dtype: int64
Casado(a)        5092
Soltero(a)       4742
Unión libre      2185
Divorciado(a)     515
Viudo(a)          130
Name: es

Las categorias observadas se revisan con el diccionario. Los valores originales se conservan para que todos los notebooks usen las mismas etiquetas. Ahora revisamos la temperatura, que es la variable numerica.


In [7]:
print(df['temperatura'].value_counts())
temperaturas_invalidas = df.loc[
    ~df['temperatura'].isin([30, 55, 80]), ['id_fila', 'temperatura']
]
print(f'Registros con temperatura fuera del diccionario: {len(temperaturas_invalidas)}')


80    6513
55    3836
30    2315
Name: temperatura, dtype: int64
Registros con temperatura fuera del diccionario: 0


**Arreglo de datos**

La variable vehiculo tiene cerca del 99.15% de datos vacios. No es adecuado asignar la misma categoria a casi todos los registros, por lo que decidimos eliminarla de los predictores. Aplicamos la misma regla a la copia de los datos de prueba.


In [8]:
print('Valores diferentes en tiempo_cupon_mayor_5min:', df['tiempo_cupon_mayor_5min'].nunique(dropna=False))
print(pd.crosstab(df['misma_direccion'], df['direccion_opuesta']))


Valores diferentes en tiempo_cupon_mayor_5min: 1
direccion_opuesta    No    Sí
misma_direccion              
No                    0  9944
Sí                 2720     0


El tiempo mayor a 5 minutos tiene el mismo valor en todas las filas, por lo que no ayuda a distinguir los registros. La variable direccion_opuesta repite la informacion de misma_direccion de forma inversa. Eliminamos estas dos variables y conservamos misma_direccion. Aplicamos la misma regla a los datos de prueba.


In [9]:
columnas_eliminar = ['vehiculo', 'tiempo_cupon_mayor_5min', 'direccion_opuesta']
df.drop(columns=columnas_eliminar, inplace=True)
df_test.drop(columns=columnas_eliminar, inplace=True)


Las variables de frecuencia conservaran sus valores nulos en este archivo. La moda se calculara dentro del pipeline de cada modelo usando solo los datos de entrenamiento. El escalamiento y la codificacion tambien se realizaran en esos pipelines.


In [10]:
((df.isnull().sum()/df.shape[0])).sort_values(ascending=False)


frecuencia_cafeteria               0.017135
frecuencia_restaurante_20_a_50     0.014924
frecuencia_comida_para_llevar      0.011924
frecuencia_restaurante_menos_20    0.010265
frecuencia_bar                     0.008449
destino                            0.000000
acompanante                        0.000000
acepta_cupon                       0.000000
misma_direccion                    0.000000
tiempo_cupon_mayor_25min           0.000000
tiempo_cupon_mayor_15min           0.000000
ingreso                            0.000000
ocupacion                          0.000000
educacion                          0.000000
tiene_hijos                        0.000000
estado_civil                       0.000000
edad                               0.000000
genero                             0.000000
vigencia_cupon                     0.000000
cupon                              0.000000
hora                               0.000000
temperatura                        0.000000
clima                           

Ahora vamos a eliminar las copias adicionales. Conservamos el ultimo registro de cada grupo y dejamos intactas las filas del archivo de prueba.


In [11]:
columnas_comparar = [col for col in df.columns if col != 'id_fila']
filas_antes = len(df)
df.drop_duplicates(subset=columnas_comparar, keep='last', inplace=True)
n_duplicados = int(df.duplicated(subset=columnas_comparar).sum())
print(f'Filas eliminadas: {filas_antes - len(df)}')
print(f'Copias adicionales restantes: {n_duplicados}')
print(f'Filas disponibles: {len(df)}')


Filas eliminadas: 74
Copias adicionales restantes: 0
Filas disponibles: 12590


Revisamos la cantidad de aceptaciones y rechazos por tipo de cupon. Cada notebook de modelos filtrara el cupon que le corresponde.


In [12]:
df.groupby('cupon')['acepta_cupon'].value_counts()


cupon               acepta_cupon
Bar                 No              1186
                    Sí               824
Cafetería           No              1996
                    Sí              1983
Comida para llevar  Sí              1720
                    No               624
Restaurante(20-50)  No               831
                    Sí               657
Restaurante(<20)    Sí              1959
                    No               810
Name: acepta_cupon, dtype: int64

Antes de exportar verificamos el identificador, las etiquetas y que la copia de prueba conserve la cantidad y el orden de sus filas.


In [13]:
assert df['id_fila'].is_unique
assert not df['id_fila'].isna().any()
assert not df['acepta_cupon'].isna().any()
assert n_duplicados == 0
assert len(df_test) == len(datos_test)
assert df_test.index.equals(datos_test.index)
print(f'Datos preparados: {df.shape}')
print(f'Datos de prueba preparados: {df_test.shape}')
assert not set(columnas_eliminar).intersection(df.columns)
assert not set(columnas_eliminar).intersection(df_test.columns)


Datos preparados: (12590, 24)
Datos de prueba preparados: (20, 22)


**Exportacion de datos**

Guardamos los datos preparados para que todos los modelos usen el mismo archivo. Los nulos pendientes se resolveran en sus pipelines. La copia de prueba se guarda sin etiquetas ni cambios en el orden. Los Excel originales se conservan para la entrega final.


In [14]:
ruta_salida = Path('data/procesados')
ruta_salida.mkdir(parents=True, exist_ok=True)
df.to_csv(ruta_salida / 'datos_limpios.csv', index=False, encoding='utf-8-sig')
df_test.to_csv(ruta_salida / 'datos_prueba_preparados.csv', index=False, encoding='utf-8-sig')
print('Archivos guardados en data/procesados')


Archivos guardados en data/procesados


## Uso de herramientas de IA generativa

Se uso Codex como apoyo para revisar y corregir la preparacion de datos.

Prompt resumido: revisar la limpieza y corregir los duplicados, el identificador, la imputacion y la exportacion, manteniendo explicaciones cortas y sin tildes. Tambien se solicito excluir la variable constante tiempo_cupon_mayor_5min y la variable redundante direccion_opuesta.

Analisis critico por completar:

- Que partes de la correccion fueron utiles y como se verificaron.
- Que limitaciones se encontraron y que decisiones se cambiaron.

Aportes propios: la carga, la exploracion inicial y la revision de los datos fueron realizadas por el estudiante. Completar con los cambios adicionales y los aprendizajes obtenidos.
